In [3]:
# 04_patient_asynchrony.ipynb – Patient‑Ventilator Asynchrony 

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================================
# Improved model: P_aw is solved directly, not reconstructed
# ============================================================

def asynchrony_ode(t, y, R, C, P_insp, PEEP, insp_time, rate, P_muscle_func):
    V = y[0]
    period = 60 / rate
    t_mod = t % period
    P_machine = P_insp if t_mod < insp_time else 0.0
    P_muscle = P_muscle_func(t)
    P_aw = PEEP + P_machine + P_muscle   # Total airway pressure (relative to atmosphere)
    Q = (P_aw - V/C) / R
    if V <= 0 and Q < 0:
        Q = 0
    return [Q]

def simulate_asynchrony(R, C, P_insp, PEEP, rate, insp_frac,
                        muscle_amp, muscle_freq, muscle_phase, trigger_threshold, duration=10):
    insp_time = (60/rate) * insp_frac
    def P_muscle(t):
        return muscle_amp * np.sin(2 * np.pi * muscle_freq * t + muscle_phase)
    
    t_eval = np.linspace(0, duration, 2000)
    sol = solve_ivp(asynchrony_ode, (0, duration), [0.0], t_eval=t_eval,
                    args=(R, C, P_insp, PEEP, insp_time, rate, P_muscle),
                    method='RK45', rtol=1e-6)
    t = sol.t
    V = sol.y[0]
    Q = np.gradient(V, t)
    # Recompute P_aw from the ODE definition (consistent)
    P_aw = np.zeros_like(t)
    for i, ti in enumerate(t):
        period = 60/rate
        t_mod = ti % period
        P_machine = P_insp if t_mod < insp_time else 0.0
        P_muscle_val = muscle_amp * np.sin(2 * np.pi * muscle_freq * ti + muscle_phase)
        P_aw[i] = PEEP + P_machine + P_muscle_val
    # Also compute machine and muscle separately for plotting
    P_machine_arr = np.array([P_insp if ((ti % (60/rate)) < insp_time) else 0.0 for ti in t])
    P_muscle_arr = muscle_amp * np.sin(2 * np.pi * muscle_freq * t + muscle_phase)
    
    # Detect asynchrony
    wasted, double = detect_asynchrony(t, V, Q, P_muscle_arr, trigger_threshold)
    return t, V, Q, P_aw, P_machine_arr, P_muscle_arr, wasted, double

def detect_asynchrony(t, V, Q, P_muscle, trigger_threshold):
    # Find peaks in P_muscle above threshold
    peaks = []
    for i in range(1, len(P_muscle)-1):
        if P_muscle[i] > trigger_threshold and P_muscle[i] > P_muscle[i-1] and P_muscle[i] >= P_muscle[i+1]:
            peaks.append(t[i])
    # Find zero crossings of flow (start of inspiration)
    zero_crossings = []
    for i in range(1, len(Q)):
        if Q[i-1] <= 0 and Q[i] > 0:
            zero_crossings.append(t[i])
    wasted = 0
    for pt in peaks:
        if not any(abs(zc - pt) < 0.2 for zc in zero_crossings):
            wasted += 1
    double = 0
    for i in range(1, len(zero_crossings)):
        if zero_crossings[i] - zero_crossings[i-1] < 0.5:
            double += 1
    return wasted, double

def compute_metrics(t, V, Q, P_aw, P_muscle, PEEP, rate, wasted, double, duration):
    V_tidal = np.max(V) - np.min(V)
    minute_vent = V_tidal * rate
    peak_P = np.max(P_aw)
    mean_P = np.mean(P_aw)
    C_dyn = V_tidal / (peak_P - PEEP) if (peak_P - PEEP) > 0 else 0
    # Work of breathing (area under PV loop) – use trapezoid
    WOB = np.trapezoid(P_aw, V) if len(V)>1 else 0
    asynch_index = (wasted / duration) * 60
    return {
        'Vt_mL': V_tidal * 1000,
        'Minute_vent_Lmin': minute_vent,
        'Peak_P_cmH2O': peak_P,
        'Mean_P_cmH2O': mean_P,
        'C_dyn_mL_cmH2O': C_dyn * 1000,
        'WOB_cmH2O_L': WOB,
        'Wasted_efforts_per_min': asynch_index,
        'Double_triggers': double
    }

def plot_asynchrony(t, V, Q, P_aw, P_machine, P_muscle, R, C, P_insp, PEEP, rate, insp_frac,
                    muscle_amp, muscle_freq, wasted, double):
    duration = t[-1] - t[0]
    metrics = compute_metrics(t, V, Q, P_aw, P_muscle, PEEP, rate, wasted, double, duration)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    ax1, ax2 = axes[0, 0], axes[0, 1]
    ax3, ax4 = axes[1, 0], axes[1, 1]
    
    # Pressure components
    ax1.plot(t, P_aw, 'b-', lw=2, label='Airway pressure')
    ax1.plot(t, P_machine + PEEP, 'g--', lw=1.5, label='Machine + PEEP')
    ax1.plot(t, P_muscle, 'r:', lw=1.5, label='Muscle pressure')
    ax1.axhline(PEEP, color='gray', linestyle='-', alpha=0.5)
    ax1.set_ylabel('Pressure (cmH2O)')
    ax1.set_title('Pressure Components')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Flow
    ax2.plot(t, Q, 'g-', lw=2)
    ax2.axhline(0, color='k', lw=0.5)
    ax2.set_ylabel('Flow (L/s)')
    ax2.set_title('Flow Waveform')
    ax2.grid(True, alpha=0.3)
    
    # Volume
    ax3.plot(t, V*1000, 'r-', lw=2)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Volume (mL)')
    ax3.set_title('Lung Volume')
    ax3.grid(True, alpha=0.3)
    
    # Pressure‑Volume loop (now consistent)
    ax4.plot(P_aw, V*1000, 'purple', lw=1.5)
    ax4.set_xlabel('Pressure (cmH2O)')
    ax4.set_ylabel('Volume (mL)')
    ax4.set_title('Pressure-Volume Loop')
    ax4.grid(True, alpha=0.3)
    
    fig.suptitle(f'Asynchrony Simulation: R={R:.1f}, C={C:.3f}, Muscle amp={muscle_amp:.1f}, freq={muscle_freq:.2f} Hz')
    
    textstr = (f"Tidal Volume: {metrics['Vt_mL']:.0f} mL\n"
               f"Minute Ventilation: {metrics['Minute_vent_Lmin']:.2f} L/min\n"
               f"Peak Pressure: {metrics['Peak_P_cmH2O']:.1f} cmH2O\n"
               f"Dynamic Compliance: {metrics['C_dyn_mL_cmH2O']:.1f} mL/cmH2O\n"
               f"Wasted efforts/min: {metrics['Wasted_efforts_per_min']:.1f}\n"
               f"Double triggers: {metrics['Double_triggers']}\n"
               f"Work of breathing: {metrics['WOB_cmH2O_L']:.2f} cmH2O·L")
    ax1.text(0.02, 0.98, textstr, transform=ax1.transAxes, fontsize=9,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Mark wasted efforts as vertical dashed lines on volume plot
    # (re‑detect peaks for plotting)
    zero_crossings = []
    for i in range(1, len(Q)):
        if Q[i-1] <= 0 and Q[i] > 0:
            zero_crossings.append(t[i])
    for i in range(1, len(P_muscle)-1):
        if P_muscle[i] > 2 and P_muscle[i] > P_muscle[i-1] and P_muscle[i] >= P_muscle[i+1]:
            if not any(abs(t[i] - zc) < 0.2 for zc in zero_crossings):
                ax3.axvline(x=t[i], color='orange', linestyle='--', alpha=0.5, linewidth=1)
    plt.show()
    
    print("\n📊 Asynchrony Metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.2f}")

# ============================================================
# Interactive widgets
# ============================================================

style = {'description_width': 'initial'}

R_slider = widgets.FloatSlider(value=8, min=2, max=30, step=0.5, description='Resistance (cmH2O/L/s):', style=style)
C_slider = widgets.FloatSlider(value=0.06, min=0.02, max=0.15, step=0.002, description='Compliance (L/cmH2O):', style=style)
P_insp_slider = widgets.FloatSlider(value=15, min=5, max=30, step=0.5, description='Insp Pressure (cmH2O):', style=style)
PEEP_slider = widgets.FloatSlider(value=5, min=0, max=15, step=1, description='PEEP (cmH2O):', style=style)
rate_slider = widgets.IntSlider(value=15, min=8, max=30, step=1, description='Rate (bpm):', style=style)
insp_frac_slider = widgets.FloatSlider(value=0.33, min=0.2, max=0.5, step=0.01, description='I:E ratio:', style=style)
muscle_amp_slider = widgets.FloatSlider(value=5, min=0, max=20, step=0.5, description='Muscle amplitude (cmH2O):', style=style)
muscle_freq_slider = widgets.FloatSlider(value=0.8, min=0.2, max=2.0, step=0.05, description='Muscle frequency (Hz):', style=style)
muscle_phase_slider = widgets.FloatSlider(value=0, min=-np.pi, max=np.pi, step=0.1, description='Muscle phase (rad):', style=style)
trigger_slider = widgets.FloatSlider(value=2, min=1, max=5, step=0.5, description='Trigger threshold (cmH2O):', style=style)
duration_slider = widgets.FloatSlider(value=10, min=5, max=30, step=1, description='Duration (s):', style=style)

def update(R, C, P_insp, PEEP, rate, insp_frac, muscle_amp, muscle_freq, muscle_phase, trigger, duration):
    clear_output(wait=True)
    t, V, Q, P_aw, P_machine, P_muscle, wasted, double = simulate_asynchrony(
        R, C, P_insp, PEEP, rate, insp_frac, muscle_amp, muscle_freq, muscle_phase, trigger, duration)
    plot_asynchrony(t, V, Q, P_aw, P_machine, P_muscle, R, C, P_insp, PEEP, rate, insp_frac,
                    muscle_amp, muscle_freq, wasted, double)

ui = widgets.VBox([
    widgets.HBox([R_slider, C_slider]),
    widgets.HBox([P_insp_slider, PEEP_slider, rate_slider, insp_frac_slider]),
    widgets.HBox([muscle_amp_slider, muscle_freq_slider, muscle_phase_slider, trigger_slider]),
    duration_slider
])

out = widgets.interactive_output(update, {
    'R': R_slider, 'C': C_slider, 'P_insp': P_insp_slider, 'PEEP': PEEP_slider,
    'rate': rate_slider, 'insp_frac': insp_frac_slider,
    'muscle_amp': muscle_amp_slider, 'muscle_freq': muscle_freq_slider,
    'muscle_phase': muscle_phase_slider, 'trigger': trigger_slider, 'duration': duration_slider
})

display(ui, out)

Output()